# Week 4 Exercise — Docstring + Unit Test Generator (profe-ssor)

Paste a **Python function** and get:
- **PEP-257 docstring** — purpose, args, returns, raises, optional example.
- **pytest unit tests** — normal cases, edge cases, and error cases.

**Stack:** Gradio UI, OpenRouter (or OpenAI) for the LLM. Set `OPENROUTER_API_KEY` in `.env` to use OpenRouter.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key:
    client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
    MODEL = "openai/gpt-4o-mini"
    print("Using OpenRouter:", MODEL)
else:
    client = OpenAI()
    MODEL = "gpt-4o-mini"
    print("Using OpenAI:", MODEL)

## Prompts

In [ ]:
DOCSTRING_SYSTEM = """You are an expert Python developer. Generate a PEP-257 style docstring for the given function.
Include: summary line, Args (with types), Returns (with type), Raises if applicable, and a short Usage example if helpful.
Output only the function with the docstring inserted (no explanation)."""

TESTS_SYSTEM = """You are a testing expert. Generate pytest unit tests for the given Python function.
Include: 2–3 normal cases, 1–2 edge cases (empty input, boundaries), and 1–2 error/exception cases where relevant.
Use assert statements. Output only the test code (import pytest and the function under test as needed). No explanation."""

In [ ]:
def generate_docstring(code):
    if not code or not code.strip():
        return ""
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": DOCSTRING_SYSTEM},
                {"role": "user", "content": code}
            ]
        )
        return (r.choices[0].message.content or "").strip()
    except Exception as e:
        return f"Error: {e}"

def generate_tests(code):
    if not code or not code.strip():
        return ""
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": TESTS_SYSTEM},
                {"role": "user", "content": code}
            ]
        )
        return (r.choices[0].message.content or "").strip()
    except Exception as e:
        return f"Error: {e}"

## Gradio UI

In [ ]:
with gr.Blocks(title="Docstring + Unit Test Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("### Paste a Python function → get docstring and pytest tests")
    code_in = gr.Textbox(label="Python function", lines=12, placeholder="def my_func(x):\n    return x + 1")
    with gr.Row():
        doc_btn = gr.Button("Generate docstring", variant="primary")
        test_btn = gr.Button("Generate unit tests")
    doc_out = gr.Code(label="Function with docstring", language="python", lines=14)
    test_out = gr.Code(label="pytest unit tests", language="python", lines=20)
    doc_btn.click(fn=generate_docstring, inputs=code_in, outputs=doc_out)
    test_btn.click(fn=generate_tests, inputs=code_in, outputs=test_out)

demo.launch()